# 02 - Exploratory analysis and statistical baseline

This notebook separates two related tasks:

- **description:** missingness, distributions and weighted prevalence;
- **association modelling:** complete-case logistic regression with robust
  standard errors.

The descriptive survey weights improve population summaries. They do not turn
the predictive model or bootstrap evaluation into full complex-survey
inference.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root() -> Path:
    candidate = Path.cwd().resolve()
    if (candidate / "pyproject.toml").exists():
        return candidate
    if (candidate.parent / "pyproject.toml").exists():
        return candidate.parent
    raise FileNotFoundError("Run this notebook from the repository root or notebooks/.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nhanes_showcase.clean import choose_weight_column
from nhanes_showcase.stats import cohort_summary, describe_missingness, summarise_subgroups

ANALYSIS_PATH = PROJECT_ROOT / "data" / "processed" / "analysis_dataset.parquet"
BASELINE_DIR = PROJECT_ROOT / "artifacts" / "baseline"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

analysis = pd.read_parquet(ANALYSIS_PATH)
weight_column = choose_weight_column(analysis)

print(f"Analysis rows: {analysis.shape[0]:,}")
print(f"Examination weight: {weight_column}")

## 1. Data types and missingness

In [ ]:
schema_view = pd.DataFrame(
    {
        "column": analysis.columns,
        "dtype": analysis.dtypes.astype(str).values,
        "non_missing": analysis.notna().sum().values,
        "unique_values": analysis.nunique(dropna=True).values,
    }
)
display(schema_view)

In [ ]:
missingness = describe_missingness(analysis)
display(missingness)
display(Image(filename=str(FIGURES_DIR / "missingness_heatmap.png"), width=850))

Missing predictive values are retained at this stage. Median and
most-frequent imputers are learned later inside each training fold, preventing
information from the validation or test sets from leaking into preprocessing.

## 2. Weighted and unweighted cohort summaries

In [ ]:
display(cohort_summary(analysis, weight_col=weight_column))

In [ ]:
for group_column in ["sex", "age_group", "race_ethnicity"]:
    display(Markdown(f"### Diabetes prevalence by `{group_column}`"))
    group_summary = summarise_subgroups(
        analysis,
        group_column,
        weight_col=weight_column,
    )
    display(group_summary)

The NHANES examination-weighted diabetes prevalence is lower than the
unweighted cohort prevalence. The weights are used for population description,
whereas the classifier is trained to predict individuals in the analytic
sample.

## 3. Distribution plots

In [ ]:
display(Image(filename=str(FIGURES_DIR / "target_balance.png"), width=650))
display(Image(filename=str(FIGURES_DIR / "bmi_by_target.png"), width=750))

## 4. Interpretable statistical baseline

The baseline is a separate complete-case `statsmodels` logistic regression.
It estimates adjusted associations, reports HC3 robust standard errors and is
not used to generate the held-out machine-learning predictions.

In [ ]:
baseline_cohort = pd.read_json(BASELINE_DIR / "baseline_cohort.json", typ="series")
display(baseline_cohort.rename("value").to_frame())

odds_ratios = pd.read_csv(BASELINE_DIR / "odds_ratios.csv")
selected_terms = odds_ratios.loc[
    odds_ratios["term"].isin(["age_years", "bmi", "C(sex)[T.Male]"]),
    ["term", "odds_ratio", "ci_low", "ci_high", "p_value"],
]
display(selected_terms.style.format(precision=3))

In [ ]:
display(Image(filename=str(FIGURES_DIR / "odds_ratios.png"), width=850))

## Interpretation boundaries

An odds ratio above one is an adjusted association with the contemporaneous
self-reported outcome. It is not a causal effect, a risk ratio or proof that
changing the predictor would change diabetes status.

Continue to `03_modelling_and_evaluation.ipynb` for leakage-safe modelling,
calibration, thresholding and held-out evaluation.